# Description

Analyzes drug-disease prediction differences between the gene-based and module-based approaches using the ARCHS4 CLAMP model.

For each drug-disease pair in the PharmacotherapyDB gold standard, it compares the standardized scores from both methods and identifies pairs where the methods disagree (different signs). It focuses on cardiovascular diseases and the Niacin case discussed in the PhenoPlier manuscript.

**Inputs** (from `021-prediction_performance.ipynb`):
- `predictions_results_aggregated.pkl`: averaged prediction scores per (trait, drug, method)

**Category note**: `indications-verbose.tsv` (PharmacotherapyDB) does not include the DM/SYM/NOT categorization from PhenoPlier's `pharmacotherapydb-v1.0/indications.tsv`. Category is approximated from `gold_standard.pkl`: `true_class=1 → DM`, `true_class=0 → NOT`.

# Module loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings

In [3]:
DATA_DIR = here('data/archs4/drug_diseases_associations')
assert DATA_DIR.exists()

PREDICTIONS_DIR = here('output/drug_disease_analyses') / 'lincs' / 'predictions'
assert PREDICTIONS_DIR.exists()

# Data loading

## PharmacotherapyDB

### Gold standard set

In [4]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


### Drug and disease name lookup

Using `indications-verbose.tsv` from [PharmacotherapyDB](https://github.com/dhimmel/indications).
This file has one row per (doid_code, drugbank_id, resource) — we deduplicate to get unique drug/disease names.
Category (DM/SYM/NOT) is approximated from `true_class` in the gold standard.

In [5]:
indications = pd.read_csv(
    DATA_DIR / 'indications-verbose.tsv',
    sep='\t',
    usecols=['doid_code', 'drugbank_id', 'drugbank_name', 'doid_name'],
).drop_duplicates(subset=['doid_code', 'drugbank_id'])

print(f'Indications shape (after dedup): {indications.shape}')
display(indications.head())

Indications shape (after dedup): (13934, 4)


,doid_code,drugbank_id,drugbank_name,doid_name
0,DOID:0014667,DB00091,Cyclosporine,disease of metabolism
1,DOID:0014667,DB00117,L-Histidine,disease of metabolism
2,DOID:0014667,DB00121,Biotin,disease of metabolism
3,DOID:0014667,DB00151,L-Cysteine,disease of metabolism
4,DOID:0014667,DB00165,Pyridoxine,disease of metabolism


In [6]:
# Build gold_standard_info: gold_standard + disease/drug names + category
gold_standard_info = gold_standard.copy()
gold_standard_info['category'] = gold_standard_info['true_class'].map({1: 'DM', 0: 'NOT'})

# Add disease and drug names from indications-verbose
name_lookup = indications.set_index(['doid_code', 'drugbank_id'])

gold_standard_info = gold_standard_info.join(
    name_lookup.rename(columns={'doid_name': 'disease', 'drugbank_name': 'drug_name'}),
    on=['trait', 'drug'],
)

# Fill any missing names with the raw IDs
gold_standard_info['disease'] = gold_standard_info['disease'].fillna(gold_standard_info['trait'])
gold_standard_info['drug_name'] = gold_standard_info['drug_name'].fillna(gold_standard_info['drug'])

display(gold_standard_info.shape)
display(gold_standard_info.head())

(998, 6)

,trait,drug,true_class,category,drug_name,disease
0,DOID:10652,DB00843,1,DM,Donepezil,Alzheimer's disease
1,DOID:10652,DB00674,1,DM,Galantamine,Alzheimer's disease
2,DOID:10652,DB01043,1,DM,Memantine,Alzheimer's disease
3,DOID:10652,DB00989,1,DM,Rivastigmine,Alzheimer's disease
4,DOID:10652,DB00810,0,NOT,Biperiden,Alzheimer's disease


## Prediction results (aggregated)

In [7]:
predictions_avg = pd.read_pickle(PREDICTIONS_DIR / 'predictions_results_aggregated.pkl')
display(predictions_avg.shape)
display(predictions_avg.head())

(1370, 5)

,trait,drug,method,score,true_class
0,DOID:0050741,DB00215,Gene-based,316134.3,1.0
1,DOID:0050741,DB00215,Module-based,353878.5,1.0
2,DOID:0050741,DB00704,Gene-based,387103.6,1.0
3,DOID:0050741,DB00704,Module-based,393946.6,1.0
4,DOID:0050741,DB00822,Gene-based,409053.0,1.0


### Merge with gold standard info

In [8]:
pharmadb_predictions = pd.merge(
    gold_standard_info,
    predictions_avg,
    on=['trait', 'drug'],
    how='inner',
)

In [9]:
pharmadb_predictions = pharmadb_predictions[
    ['trait', 'drug', 'disease', 'drug_name', 'method', 'score', 'true_class_x', 'category']
].rename(columns={'true_class_x': 'true_class'})

display(pharmadb_predictions.shape)
assert pharmadb_predictions.shape[0] == predictions_avg.shape[0]
display(pharmadb_predictions.head())

(1370, 8)

,trait,drug,disease,drug_name,method,score,true_class,category
0,DOID:10652,DB00843,Alzheimer's disease,Donepezil,Gene-based,344883.3,1,DM
1,DOID:10652,DB00843,Alzheimer's disease,Donepezil,Module-based,379847.9,1,DM
2,DOID:10652,DB00674,Alzheimer's disease,Galantamine,Gene-based,359189.6,1,DM
3,DOID:10652,DB00674,Alzheimer's disease,Galantamine,Module-based,367012.0,1,DM
4,DOID:10652,DB01043,Alzheimer's disease,Memantine,Gene-based,280545.4,1,DM


In [10]:
print('Unique diseases:', pharmadb_predictions['disease'].nunique())
print('Unique drugs:', pharmadb_predictions['drug'].nunique())

Unique diseases: 71
Unique drugs: 337


In [11]:
# Compute score statistics from the non-aggregated predictions (all thresholds),
# where scores are ranks within the full DOID-mapped prediction set (not just the
# 685 gold-standard pairs), so standardization reflects the global distribution.
_predictions_raw = pd.read_pickle(PREDICTIONS_DIR / 'predictions_results.pkl')
data_stats = _predictions_raw.groupby('method', observed=True)['score'].describe()
display(data_stats)

,count,mean,std,min,25%,50%,75%,max
method,,,,,,,,
Gene-based,167825.0,240697.311289,138066.913676,35.5,118048.0,259369.0,371597.0,425879.0
Module-based,167825.0,241154.569954,134424.439248,17.5,124130.0,260512.0,365440.0,425880.0


# Standardize scores for each method

In [12]:
def _standardize(x):
    return (x['score'] - data_stats.loc[x['method'], 'mean']) / data_stats.loc[x['method'], 'std']


pharmadb_predictions = pharmadb_predictions.assign(
    score_std=pharmadb_predictions.apply(_standardize, axis=1)
)

display(pharmadb_predictions.head())

,trait,drug,disease,drug_name,method,score,true_class,category,score_std
0,DOID:10652,DB00843,Alzheimer's disease,Donepezil,Gene-based,344883.3,1,DM,0.754605
1,DOID:10652,DB00843,Alzheimer's disease,Donepezil,Module-based,379847.9,1,DM,1.031757
2,DOID:10652,DB00674,Alzheimer's disease,Galantamine,Gene-based,359189.6,1,DM,0.858224
3,DOID:10652,DB00674,Alzheimer's disease,Galantamine,Module-based,367012.0,1,DM,0.936269
4,DOID:10652,DB01043,Alzheimer's disease,Memantine,Gene-based,280545.4,1,DM,0.288614


In [13]:
# Sanity check: standardized scores should be zero-mean, unit-variance per method
_tmp = pharmadb_predictions.groupby('method')[['score', 'score_std']].describe()
display(_tmp)

score                                                   \
              count           mean           std       min       25%   
method                                                                 
Gene-based    685.0  377148.662920  46067.093080  138065.6  356974.6   
Module-based  685.0  381682.431971  36004.177391  234996.1  362150.4   

                                           score_std                      \
                   50%       75%       max     count      mean       std   
method                                                                     
Gene-based    390004.2  412757.8  425826.0     685.0  0.988299  0.333658   
Module-based  389430.0  410034.4  425856.2     685.0  1.045404  0.267840   

                                                                
                   min       25%       50%       75%       max  
method                                                          
Gene-based   -0.743348  0.842181  1.081410  1.246211  1.340862  
Module-based -0.045814  0.900103  1.103039  1.256318  1.374018

# List diseases

In [14]:
pharmadb_predictions['disease'].unique()

array(["Alzheimer's disease", "Barrett's esophagus", "Crohn's disease",
       "Parkinson's disease", 'acquired immunodeficiency syndrome',
       'alcohol dependence', 'allergic rhinitis', 'anemia', 'DOID:2355',
       'ankylosing spondylitis', 'asthma', 'atherosclerosis', 'DOID:1936',
       'DOID:184', 'brain cancer', 'breast cancer', 'DOID:1612',
       'cervical cancer', 'DOID:784',
       'chronic obstructive pulmonary disease', 'coronary artery disease',
       'DOID:3393', 'epilepsy syndrome', 'DOID:1826', 'esophageal cancer',
       'gestational diabetes', 'DOID:1686', 'glaucoma', 'gout',
       'DOID:2531', 'hypertension', 'DOID:10763', 'hypothyroidism',
       'DOID:263', 'kidney cancer', 'liver cancer', 'DOID:1324',
       'lung cancer', 'malaria', 'DOID:12365', 'melanoma', 'migraine',
       'multiple sclerosis', 'nephrolithiasis', 'obesity',
       'osteoarthritis', 'osteoporosis', 'DOID:1793', 'pancreatic cancer',
       'pancreatitis', 'DOID:824', 'prostate cancer', 'DO

# Look for differences in scores between methods

For each drug-disease pair, identify where gene-based and module-based methods give opposite signs (one above mean, one below mean). These are the most interesting cases where the latent variable structure changes the prediction.

In [15]:
def _compare(x):
    """
    For a drug-disease pair with two rows (one per method), compute:
    - different_sign: whether the two standardized scores have opposite signs
    - score_difference: absolute difference between the two standardized scores
    """
    assert x.shape[0] == 2
    x_sign = np.sign(x['score_std'].values)
    x0 = x.iloc[0]['score_std']
    x1 = x.iloc[1]['score_std']
    return pd.Series(
        {'different_sign': x_sign[0] != x_sign[1], 'score_difference': np.abs(x0 - x1)}
    )

In [16]:
pharmadb_predictions = pharmadb_predictions.set_index(['trait', 'drug']).join(
    pharmadb_predictions.groupby(['trait', 'drug']).apply(_compare, include_groups=False)
)

display(pharmadb_predictions.head())

disease    drug_name        method     score  \
trait      drug                                                                
DOID:10652 DB00843  Alzheimer's disease    Donepezil    Gene-based  344883.3   
           DB00843  Alzheimer's disease    Donepezil  Module-based  379847.9   
           DB00674  Alzheimer's disease  Galantamine    Gene-based  359189.6   
           DB00674  Alzheimer's disease  Galantamine  Module-based  367012.0   
           DB01043  Alzheimer's disease    Memantine    Gene-based  280545.4   

                    true_class category  score_std  different_sign  \
trait      drug                                                      
DOID:10652 DB00843           1       DM   0.754605           False   
           DB00843           1       DM   1.031757           False   
           DB00674           1       DM   0.858224           False   
           DB00674           1       DM   0.936269           False   
           DB01043           1       DM   0.288614           False   

                    score_difference  
trait      drug                       
DOID:10652 DB00843          0.277152  
           DB00843          0.277152  
           DB00674          0.078045  
           DB00674          0.078045  
           DB01043          0.613697

## Across all diseases

In [17]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'max_colwidth', None):
    _tmp = pharmadb_predictions[pharmadb_predictions['different_sign']].sort_values(
        ['score_difference', 'drug_name', 'method'], ascending=[False, False, False]
    )
    display(_tmp.shape)
    display(_tmp)

(32, 9)

disease             drug_name        method  \
trait      drug                                                                
DOID:2998  DB00445     testicular cancer            Epirubicin  Module-based   
           DB00445     testicular cancer            Epirubicin    Gene-based   
           DB00997     testicular cancer           Doxorubicin  Module-based   
           DB00997     testicular cancer           Doxorubicin    Gene-based   
DOID:10763 DB00195          hypertension             Betaxolol  Module-based   
           DB00195          hypertension             Betaxolol    Gene-based   
DOID:10283 DB01248       prostate cancer             Docetaxel  Module-based   
           DB01248       prostate cancer             Docetaxel    Gene-based   
DOID:10763 DB00960          hypertension              Pindolol  Module-based   
           DB00960          hypertension              Pindolol    Gene-based   
           DB00691          hypertension             Moexipril  Module-based   
           DB00691          hypertension             Moexipril    Gene-based   
           DB00310          hypertension        Chlorthalidone  Module-based   
           DB00310          hypertension        Chlorthalidone    Gene-based   
           DB00373          hypertension               Timolol  Module-based   
           DB00373          hypertension               Timolol    Gene-based   
DOID:10283 DB00537       prostate cancer         Ciprofloxacin  Module-based   
           DB00537       prostate cancer         Ciprofloxacin    Gene-based   
DOID:10763 DB00945          hypertension  Acetylsalicylic acid  Module-based   
           DB00945          hypertension  Acetylsalicylic acid    Gene-based   
DOID:1612  DB00014         breast cancer             Goserelin  Module-based   
           DB00014         breast cancer             Goserelin    Gene-based   
DOID:418   DB00722  systemic scleroderma            Lisinopril  Module-based   
           DB00722  systemic scleroderma            Lisinopril    Gene-based   
DOID:9206  DB00338   Barrett's esophagus            Omeprazole  Module-based   
           DB00338   Barrett's esophagus            Omeprazole    Gene-based   
           DB00736   Barrett's esophagus          Esomeprazole  Module-based   
           DB00736   Barrett's esophagus          Esomeprazole    Gene-based   
DOID:9970  DB00273               obesity            Topiramate  Module-based   
           DB00273               obesity            Topiramate    Gene-based   
DOID:418   DB00425  systemic scleroderma              Zolpidem  Module-based   
           DB00425  systemic scleroderma              Zolpidem    Gene-based   

                       score  true_class category  score_std  different_sign  \
trait      drug                                                                
DOID:2998  DB00445  408208.3           1       DM   1.242733            True   
           DB00445  204920.7           1       DM  -0.259125            True   
           DB00997  408208.3           1       DM   1.242733            True   
           DB00997  204920.7           1       DM  -0.259125            True   
DOID:10763 DB00195  382730.8           1       DM   1.053203            True   
           DB00195  189386.8           1       DM  -0.371635            True   
DOID:10283 DB01248  400079.8           1       DM   1.182264            True   
           DB01248  217953.8           1       DM  -0.164728            True   
DOID:10763 DB00960  381314.4           1       DM   1.042666            True   
           DB00960  226107.0           1       DM  -0.105676            True   
           DB00691  293118.8           1       DM   0.386568            True   
           DB00691  138065.6           1       DM  -0.743348            True   
           DB00310  344477.3           1       DM   0.768631            True   
           DB00310  201773.9           1       DM  -0.281917            True   
           DB00373  352701.4    

In [18]:
def find_differences(trait_name):
    """Show drug-disease pairs for a given disease where methods disagree."""
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'max_colwidth', None):
        _tmp = pharmadb_predictions[
            (pharmadb_predictions['disease'] == trait_name)
            & (pharmadb_predictions['different_sign'])
        ].sort_values(
            ['score_difference', 'drug_name', 'method'], ascending=[False, False, False]
        )
        display(_tmp)

# Cardiovascular diseases

Niacin and cardiovascular traits — a key case study in the PhenoPlier manuscript.

## Coronary artery disease

In [19]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'max_colwidth', None):
    _tmp = pharmadb_predictions[
        (pharmadb_predictions['drug_name'] == 'Niacin')
        & (pharmadb_predictions['disease'] == 'coronary artery disease')
    ].sort_values(
        ['score_difference', 'drug_name', 'method'], ascending=[False, False, False]
    )
    display(_tmp.head(50))

disease drug_name        method     score  \
trait     drug                                                                 
DOID:3393 DB00627  coronary artery disease    Niacin  Module-based  402970.4   
          DB00627  coronary artery disease    Niacin    Gene-based  410337.2   

                   true_class category  score_std  different_sign  \
trait     drug                                                      
DOID:3393 DB00627           1       DM   1.203768           False   
          DB00627           1       DM   1.228679           False   

                   score_difference  
trait     drug                       
DOID:3393 DB00627          0.024911  
          DB00627          0.024911

In [20]:
find_differences('coronary artery disease')

,,disease,drug_name,method,score,true_class,category,score_std,different_sign,score_difference
trait,drug,,,,,,,,,


## Atherosclerosis

In [21]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'max_colwidth', None):
    _tmp = pharmadb_predictions[
        (pharmadb_predictions['drug_name'] == 'Niacin')
        & (pharmadb_predictions['disease'] == 'atherosclerosis')
    ].sort_values(
        ['score_difference', 'drug_name', 'method'], ascending=[False, False, False]
    )
    display(_tmp.head(50))

disease drug_name        method     score  \
trait     drug                                                         
DOID:1936 DB00627  atherosclerosis    Niacin  Module-based  331325.6   
          DB00627  atherosclerosis    Niacin    Gene-based  358965.6   

                   true_class category  score_std  different_sign  \
trait     drug                                                      
DOID:1936 DB00627           1       DM   0.670793           False   
          DB00627           1       DM   0.856601           False   

                   score_difference  
trait     drug                       
DOID:1936 DB00627          0.185808  
          DB00627          0.185808

In [22]:
find_differences('atherosclerosis')

,,disease,drug_name,method,score,true_class,category,score_std,different_sign,score_difference
trait,drug,,,,,,,,,
